In [ ]:
import pandas as pd
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models
import torch.nn as nn
from torch.utils.data import ConcatDataset
import torch.optim as optim
import random
from torch.utils.data import TensorDataset
from sklearn.cluster import KMeans
from torch.nn.utils import parameters_to_vector
from tqdm import tqdm
import time
import copy

dataset : https://www.kaggle.com/datasets/puneet6060/intel-image-classification?select=seg_train

Create the label files

In [ ]:
dataset_train_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_train/seg_train'
dataset_test_path = '/Users/laurageneaulibourne/Downloads/S4/ia/exam/dataset/seg_test/seg_test'

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}

for c in classe.keys():
    label_train = []
    label_test = []

    class_train_path = os.path.join(dataset_train_path, c)
    class_test_path = os.path.join(dataset_test_path, c)

    img_train_path = class_train_path +'/img'
    img_test_path = class_test_path+'/img'

    for imag in os.listdir(img_train_path):
        if not imag.startswith('.'): # avoid .DS_Store
            label_train.append({'image_name' : imag, 'label' : classe[c]})


    for imag in os.listdir(img_test_path):
        if not imag.startswith('.'):
            label_test.append({'image_name' : imag, 'label' : classe[c]})


    df_train = pd.DataFrame(label_train)
    df_train.to_csv(os.path.join(class_train_path, f"{classe[c]}_label_train.csv"), index=False)
    df_test = pd.DataFrame(label_test)
    df_test.to_csv(os.path.join(class_test_path, f"{classe[c]}_label_test.csv"), index=False)



create our dataset

In [ ]:
#For each task we will do a classification on 2 classes, so we will create a new dataset with only 2 classes by merging our 2 directories and same thing for the file with the labels.

def task(A,B, idx):
    '''A and B are the 2 classes of the idx task'''
    path_img_A_train = os.path.join(dataset_train_path, A, 'img')
    path_img_B_train = os.path.join(dataset_train_path, B, 'img')
    path_img_AB_train = os.path.join(dataset_train_path, f'df_{A}_{B}', 'img')
    path_img_A_test = os.path.join(dataset_test_path, A, 'img')
    path_img_B_test = os.path.join(dataset_test_path, B, 'img')
    path_img_AB_test = os.path.join(dataset_test_path, f'df_{A}_{B}', 'img')

    #create the directory if it does not exist
    os.makedirs(path_img_AB_train, exist_ok=True)
    os.makedirs(path_img_AB_test, exist_ok=True)

    def move(init, final):
        for file in os.listdir(init):
            if not file.startswith('.'):
                source_file = os.path.join(init, file)
                dest_file = os.path.join(final, file)
                os.rename(source_file, dest_file)



    move(path_img_A_train, path_img_AB_train)
    move(path_img_B_train, path_img_AB_train)

    move(path_img_A_test, path_img_AB_test)
    move(path_img_B_test, path_img_AB_test)

    label_A_train = pd.read_csv(os.path.join(dataset_train_path, A, f"{classe[A]}_label_train.csv"))
    label_B_train = pd.read_csv(os.path.join(dataset_train_path, B, f"{classe[B]}_label_train.csv"))
    label_A_train['label'] = 0
    label_B_train['label'] = 1
    label_AB_train = pd.concat([label_A_train, label_B_train], ignore_index=True)
    label_AB_train['task_id'] = idx
    label_AB_train.to_csv(os.path.join(dataset_train_path,f'df_{A}_{B}', 'label_train.csv'), index=False)



    label_A_test = pd.read_csv(os.path.join(dataset_test_path, A, f"{classe[A]}_label_test.csv"))
    label_B_test = pd.read_csv(os.path.join(dataset_test_path, B, f"{classe[B]}_label_test.csv"))
    label_A_test['label'] = 0
    label_B_test['label'] = 1
    label_AB_test = pd.concat([label_A_test, label_B_test], ignore_index=True)
    label_AB_test['task_id'] = idx
    label_AB_test.to_csv(os.path.join(dataset_test_path, f'df_{A}_{B}', 'label_test.csv'), index=False)

    return (path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test)

path_img_AB_train, path_img_AB_test, label_AB_train, label_AB_test = task('buildings', 'street', 0)
path_img_CD_train, path_img_CD_test, label_CD_train, label_CD_test = task('glacier', 'mountain', 1)

EmptyDataError: No columns to parse from file

Repartition of the dataset through the 6 classes

In [ ]:

def repartition(label_train, label_test, classe_0, classe_1):

    train_tot = len(label_train) # total number of images in the train for this task
    test_tot  = len(label_test) # total number of images in the test for this task

    nb_0_train = (label_train['label'] == 0).sum() #number of images in the train for the class 0
    nb_1_train = (label_train['label'] == 1).sum() #number of images in the train for the class 1
    nb_0_test  = (label_test['label'] == 0).sum() #number of images in the test for the class 0
    nb_1_test  = (label_test['label'] == 1).sum() #number of images in the test for the class 1

    print(f"Task : {label_train['task_id'].iloc[0]} \n")
    print(f" Total images in the train for the task {classe_0}/{classe_1} : {train_tot}")
    print(f" Pourcentage of images in the train for the class {classe_0} : {nb_0_train * 100/ train_tot}%")
    print(f" Pourcentage of images in the train for the class {classe_1} : {nb_1_train * 100/ train_tot}%")

    print(f" Total images in the test for the task {classe_0}/{classe_1} : {test_tot}")
    print(f" Pourcentage of images in the test for the class {classe_0} : {nb_0_test * 100/ test_tot}%")
    print(f" Pourcentage of images in the test for the class {classe_1} : {nb_1_test * 100/ test_tot}% \n")


repartition(label_AB_train, label_AB_test, 'buildings', 'street')
repartition(label_CD_train, label_CD_test, 'glacier', 'mountain')



Task : 0 

 Total images in the train for the task buildings/street : 4573
 Pourcentage of images in the train for the class buildings : 47.91165536846709%
 Pourcentage of images in the train for the class street : 52.08834463153291%
 Total images in the test for the task buildings/street : 938
 Pourcentage of images in the test for the class buildings : 46.588486140724946%
 Pourcentage of images in the test for the class street : 53.411513859275054% 

Task : 1 

 Total images in the train for the task glacier/mountain : 4916
 Pourcentage of images in the train for the class glacier : 48.901545972335235%
 Pourcentage of images in the train for the class mountain : 51.098454027664765%
 Total images in the test for the task glacier/mountain : 1078
 Pourcentage of images in the test for the class glacier : 51.298701298701296%
 Pourcentage of images in the test for the class mountain : 48.701298701298704% 



In [ ]:
class CustomImageDataset(Dataset):

    def __init__(self, annotations_file, img_dir, portion, img_transform=None): # portion is the pourcentage of the dataset that we will keep in order to avoid catastrophic forgetting during the new task.

        self.img_dir = img_dir
        self.img_transform = img_transform

        if portion < 1.0:
            self.img_labels = annotations_file.sample(frac=portion, random_state=42).reset_index(drop=True)
        else:
            self.img_labels = annotations_file

    def __len__(self): #Return the number of samples in the dataset.
        return len(self.img_labels)

    def __getitem__(self, idx): #For a given index, it returns the sample (image, label) associated to it.

        # Get the file path of the image by combining the directory and the image filename from the labels DataFrame
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])

        # Read the image from the specified file path and convert it to a tensor
        image = Image.open(img_path)

        # Retrieve the corresponding label for the image from the labels DataFrame
        label = self.img_labels.iloc[idx, 1]

        task = self.img_labels.iloc[idx, 2]

        # If a transform function is provided, apply it to the image
        if self.img_transform:
            image = self.img_transform(image)

        # Return the transformed image and label and task as a triplet
        return image, label, task


In [ ]:
# List of transformations that will be applied to images
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])


In [ ]:

dataset_train_AB_full = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = 1 , img_transform=transform)
dataset_test_AB_full = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = 1 , img_transform=transform)

train_dataloader_AB_full = DataLoader(dataset_train_AB_full, batch_size=64, shuffle=True)
test_dataloader_AB_full= DataLoader(dataset_test_AB_full, batch_size=64, shuffle=True)


portion = 0.5 #portion of the dataset of the previous dataset we want to keep for avoiding the catastrophic forgetting during the new task.



dataset_train_AB_part = CustomImageDataset(annotations_file= label_AB_train, img_dir = path_img_AB_train, portion = portion , img_transform=transform)
dataset_test_AB_part = CustomImageDataset(annotations_file= label_AB_test, img_dir = path_img_AB_test, portion = portion, img_transform=transform)

train_dataloader_AB_part = DataLoader(dataset_train_AB_part, batch_size=64, shuffle=True)
test_dataloader_AB_part= DataLoader(dataset_test_AB_part, batch_size=64,  shuffle=True)





dataset_train_CD_full = CustomImageDataset(annotations_file= label_CD_train, img_dir = path_img_CD_train, portion = 1 , img_transform=transform)
dataset_test_CD_full = CustomImageDataset(annotations_file= label_CD_test, img_dir = path_img_CD_test, portion = 1, img_transform=transform)

train_dataloader_CD_full = DataLoader(dataset_train_CD_full, batch_size=64, shuffle=True)
test_dataloader_CD_full= DataLoader(dataset_test_CD_full, batch_size=64, shuffle=True)






dataset_train_ABCD = ConcatDataset([dataset_train_AB_part, dataset_train_CD_full])
dataset_test_ABCD = ConcatDataset([dataset_test_AB_part, dataset_test_CD_full])

train_dataloader_ABCD = DataLoader(dataset_train_ABCD, batch_size=64, shuffle=True)
test_dataloader_ABCD= DataLoader(dataset_test_ABCD, batch_size=64, shuffle=True)


In [ ]:
def train_loop_basic(dataloader, model, loss, optimizer, num_epochs, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.train()

    for epoch in range(num_epochs):

        epoch_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0}  # accuracy dictionary for each class

        for image, label, _ in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            optimizer.zero_grad()  # Set gradient to zero

            # Compute prediction and loss
            pred = model(image)

            batch_loss = loss(pred, label)
            epoch_loss += batch_loss.item() * image.size(0)

            pred_class = pred.argmax(dim=1)
            pred_classA = (pred_class == 0)
            label_A = (label == 0)
            dic_acc["A_true"] += (pred_classA & label_A).sum().item() #when the prediction is equal the ground truth
            dic_acc["A_tot"] += label_A.sum().item()

            pred_classB = (pred_class == 1)
            label_B = (label == 1)
            dic_acc["B_true"] += (pred_classB & label_B).sum().item()
            dic_acc["B_tot"] += label_B.sum().item()

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        epoch_loss = epoch_loss / size


        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
    )




In [ ]:
def test_loop_basic(dataloader, model, loss, device):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.eval()

    with torch.no_grad():

        test_loss = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0, "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class

        for image, label, task in dataloader:

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)
            task = task.to(device)

            # Compute prediction and loss
            pred = model(image)

            batch_loss = loss(pred, label)
            test_loss += batch_loss.item() * image.size(0)



            pred_class = pred.argmax(dim=1)

            #Claculate the accuracy
            pred_class0_task0= (pred_class == 0) & (task==0)
            label0_task0 = (label == 0) & (task==0)
            dic_acc["A_true"] += (pred_class0_task0 & label0_task0).sum().item()
            dic_acc["A_tot"] += label0_task0.sum().item()

            pred_class1_task0= (pred_class == 1) & (task==0)
            label1_task0 = (label == 1) & (task==0)
            dic_acc["B_true"] += (pred_class1_task0 & label1_task0).sum().item()
            dic_acc["B_tot"] += label1_task0.sum().item()

            pred_class0_task1= (pred_class == 0) & (task==1)
            label0_task1 = (label == 0) & (task==1)
            dic_acc["C_true"] += (pred_class0_task1 & label0_task1).sum().item()
            dic_acc["C_tot"] += label0_task1.sum().item()

            pred_class1_task1= (pred_class == 1) & (task==1)
            label1_task1 = (label == 1) & (task==1)
            dic_acc["D_true"] += (pred_class1_task1 & label1_task1).sum().item()
            dic_acc["D_tot"] += label1_task1.sum().item()

        test_loss = test_loss / size


        print(f'Test Loss: {test_loss}, '
            f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
            f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
            f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
            f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %'
        )


In [ ]:
basic_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # we consider a pretrained model
num_features = basic_model.fc.in_features
basic_model.fc = nn.Linear(num_features, 2) #2 classes so 2 neurons

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')


loss = nn.CrossEntropyLoss()
optimizer_basic = optim.SGD(basic_model.parameters(), lr=0.001, momentum=0.9)
num_epochs = 10

A = buildings ; B = street

In [ ]:
%%time
train_loop_basic(dataloader = train_dataloader_AB_full , model = basic_model, loss = loss, optimizer = optimizer_basic, num_epochs = num_epochs, device = device)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!

In [ ]:
%%time
test_loop_basic(dataloader = test_dataloader_AB_full  , model = basic_model, loss = loss,device = device)

Test Loss: 0.0030935972865456457, Accuracy A: 92.90617848970251 %, B: 93.21357285429141 %, 


A= glacier ; B = mountain

In [ ]:
%%time
train_loop_basic(dataloader = train_dataloader_CD_full  , model = basic_model, loss = loss, optimizer = optimizer_basic, num_epochs = num_epochs, device = device)

Epoch 1/10, Total Mean Loss: 0.008167542423219386, Accuracy A: 80.9900166389351 %, B: 80.65286624203821 %, 
Epoch 2/10, Total Mean Loss: 0.002852557230046651, Accuracy A: 93.17803660565724 %, B: 93.0732484076433 %, 
Epoch 3/10, Total Mean Loss: 0.001477894398245121, Accuracy A: 97.46256239600666 %, B: 97.85031847133757 %, 
Epoch 4/10, Total Mean Loss: 0.0007773194631805645, Accuracy A: 99.25124792013311 %, B: 99.12420382165605 %, 
Epoch 5/10, Total Mean Loss: 0.0004515600560477005, Accuracy A: 99.62562396006656 %, B: 99.60191082802548 %, 
Epoch 6/10, Total Mean Loss: 0.00033676629365809064, Accuracy A: 99.79201331114808 %, B: 99.88057324840764 %, 
Epoch 7/10, Total Mean Loss: 0.0002245162023561783, Accuracy A: 99.87520798668885 %, B: 99.8407643312102 %, 
Epoch 8/10, Total Mean Loss: 0.0001805129019010045, Accuracy A: 99.91680532445923 %, B: 99.76114649681529 %, 
Epoch 9/10, Total Mean Loss: 0.00018241450256282316, Accuracy A: 99.79201331114808 %, B: 99.88057324840764 %, 
Epoch 10/10, T

In [ ]:
%%time
test_loop_basic(dataloader = test_dataloader_CD_full  , model = basic_model, loss = loss,device = device)

Test Loss: 0.006327900641592624, Accuracy A: 0.0 %, B: 0.0 %, C: 87.34177215189874 %, D: 89.9047619047619 %


A = buildings ; B = street; C= glacier ; D = mountain

In [ ]:
%%time
test_loop_basic(dataloader = test_dataloader_ABCD  , model = basic_model, loss = loss,device = device)

Test Loss: 0.010457569060205412, Accuracy A: 93.56223175965665 %, B: 42.79661016949152 %, C: 87.34177215189874 %, D: 89.9047619047619 %


In [ ]:
class rehearsal(nn.Module):
# Constructor method to initialize layers of the network def __init__(self):
        def __init__(self,num_classe=2):
                super().__init__()
                self.num_classe = num_classe
                self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) #pretrain model
                self.num_features = self.model.fc.in_features
                self.model.fc = nn.Identity()
                self.heads = nn.ModuleDict()

        def add_task(self, num_task):
                num_task = str(num_task)
                if num_task not in self.heads:
                        self.heads[num_task] = nn.Linear(self.num_features, self.num_classe) #2 tasks with 2 classes

        # Forward pass of the network
        def forward(self, x, num_task):
                features = self.model(x)
                outputs = torch.zeros(x.size(0), self.num_classe, device=x.device)

                for t in torch.unique(num_task):
                        t_str = str(t.item())
                        if t_str not in self.heads:
                                raise ValueError(f"Task {t_str} is not found in the model.")
                        mask = (num_task == t)
                        outputs[mask] = self.heads[t_str](features[mask])

                return outputs


In [ ]:
def train_loop_rehearsal(dataloader, model, loss, optimizer, num_epochs, device):

    model = model.to(device)
    model.train()

    for epoch in range(num_epochs):

        epoch_lossAB = 0.0
        epoch_lossCD = 0.0
        epoch_loss_tot = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class


        for image, label, task in dataloader:


            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0
            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)
            task = task.to(device)

            optimizer.zero_grad()  # Set gradient to zero

            # Compute prediction and loss
            pred = model(image, task)

            mask_AB = (task == 0)
            mask_CD = (task == 1)

            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]
                label_AB = label[mask_AB]

                batch_lossAB = loss(pred_AB, label_AB)
                epoch_lossAB += batch_lossAB.item()* label_AB.size(0)

                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()


            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]
                label_CD = label[mask_CD]

                batch_lossCD = loss(pred_CD, label_CD)
                epoch_lossCD += batch_lossCD.item()* label_CD.size(0)

                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()


            batch_loss_tot = batch_lossAB + batch_lossCD
            epoch_loss_tot += batch_loss_tot.item()

            # Backpropagation
            batch_loss_tot.backward()
            optimizer.step()

        nb_AB = dic_acc["A_tot"] + dic_acc["B_tot"]
        nb_CD = dic_acc["C_tot"] + dic_acc["D_tot"]

        epoch_lossAB = epoch_lossAB / nb_AB
        if nb_CD > 0:
            epoch_lossCD = epoch_lossCD / nb_CD
        else :
            epoch_lossCD = 0.0
        epoch_loss = (epoch_lossAB + epoch_lossCD)


        print(f'Epoch {epoch+1}/{num_epochs}, Total Mean Loss: {epoch_loss}, 1st task Loss: {epoch_lossAB}, 2nd task Loss: {epoch_lossCD}, '
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')




In [ ]:
def test_loop_rehearsal(dataloader, model, loss, device):

    model = model.to(device)
    model.eval()
    loss_test = 0.0
    loss_testAB = 0.0
    loss_testCD = 0.0

    dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class

    with torch.no_grad():


        for image, label, task in dataloader:

            batch_lossAB = 0.0
            batch_lossCD = 0.0
            batch_loss_tot = 0.0

            #put image and label on the same device
            image= image.to(device)
            label= label.to(device)

            # Compute prediction and loss
            pred = model(image, task)

            mask_AB = (task == 0)
            mask_CD = (task == 1)

            if mask_AB.sum() > 0: # at least one image from the 1st task in the batch
                pred_AB = pred[mask_AB]
                label_AB = label[mask_AB]

                batch_lossAB = loss(pred_AB, label_AB)
                loss_testAB += batch_lossAB.item()* label_AB.size(0)

                # For each image, choose the class with the higher score
                pred_classAB = pred_AB.argmax(dim=1)
                pred_classA = (pred_classAB == 0)
                label_A = (label_AB == 0)
                dic_acc["A_true"] += (pred_classA & label_A).sum().item()
                dic_acc["A_tot"] += label_A.sum().item()

                pred_classB = (pred_classAB == 1)
                label_B = (label_AB == 1)
                dic_acc["B_true"] += (pred_classB & label_B).sum().item()
                dic_acc["B_tot"] += label_B.sum().item()



            if mask_CD.sum() > 0: # at least one image from the 2nd task in the batch
                pred_CD = pred[mask_CD]
                label_CD = label[mask_CD]

                batch_lossCD = loss(pred_CD, label_CD)
                loss_testCD += batch_lossCD.item()* label_CD.size(0)

                pred_classCD = pred_CD.argmax(dim=1)
                pred_classC = (pred_classCD == 0)
                label_C = (label_CD == 0)
                dic_acc["C_true"] += (pred_classC & label_C).sum().item()
                dic_acc["C_tot"] += label_C.sum().item()

                pred_classD = (pred_classCD == 1)
                label_D = (label_CD == 1)
                dic_acc["D_true"] += (pred_classD & label_D).sum().item()
                dic_acc["D_tot"] += label_D.sum().item()

        nb_AB = dic_acc["A_tot"] + dic_acc["B_tot"]
        nb_CD = dic_acc["C_tot"] + dic_acc["D_tot"]

        loss_testAB = loss_testAB / nb_AB
        if nb_CD > 0:
            loss_testCD = loss_testCD / nb_CD
        else:
            loss_testCD = 0.0

        loss_test =  loss_testAB + loss_testCD


        print(f'Total Mean Loss per images: {loss_test}, 1st task Loss: {loss_testAB}, 2nd task Loss: {loss_testCD}',
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')

    return loss_testAB, dic_acc



In [ ]:
model_rehearsal = rehearsal()
model_rehearsal.add_task(0)
model_rehearsal.add_task(1)
optimizer_rehearsal = optim.SGD(model_rehearsal.parameters(), lr=0.001, momentum=0.9)

In [ ]:
%%time
train_loop_rehearsal(train_dataloader_AB_full , model_rehearsal, loss, optimizer_rehearsal, num_epochs, device)

Epoch 1/10, Total Mean Loss: 0.3057336507891764, 1st task Loss: 0.3057336507891764, 2nd task Loss: 0.0, Accuracy A: 85.44043815609311 %, B: 87.61544920235096 %, C: 0.0 %, D: 0.0 %
Epoch 2/10, Total Mean Loss: 0.12839829311892473, 1st task Loss: 0.12839829311892473, 2nd task Loss: 0.0, Accuracy A: 94.56869009584665 %, B: 96.09571788413098 %, C: 0.0 %, D: 0.0 %
Epoch 3/10, Total Mean Loss: 0.08876903486276595, 1st task Loss: 0.08876903486276595, 2nd task Loss: 0.0, Accuracy A: 96.80511182108626 %, B: 97.0193115029387 %, C: 0.0 %, D: 0.0 %
Epoch 4/10, Total Mean Loss: 0.06188387762149888, 1st task Loss: 0.06188387762149888, 2nd task Loss: 0.0, Accuracy A: 97.67229575536285 %, B: 98.65659109991604 %, C: 0.0 %, D: 0.0 %
Epoch 5/10, Total Mean Loss: 0.04207014500208504, 1st task Loss: 0.04207014500208504, 2nd task Loss: 0.0, Accuracy A: 98.81332724783204 %, B: 99.11838790931989 %, C: 0.0 %, D: 0.0 %
Epoch 6/10, Total Mean Loss: 0.02558713792091109, 1st task Loss: 0.02558713792091109, 2nd tas

In [ ]:
%%time
test_loop_rehearsal(test_dataloader_AB_full, model_rehearsal, loss, device)

Total Mean Loss per images: 0.19169316858625107, 1st task Loss: 0.19169316858625107, 2nd task Loss: 0.0 Accuracy A: 93.36384439359267 %, B: 93.6127744510978 %, C: 0.0 %, D: 0.0 %


In [ ]:
%%time
train_loop_rehearsal(train_dataloader_ABCD , model_rehearsal, loss, optimizer_rehearsal, num_epochs, device)

Epoch 1/10, Total Mean Loss: 0.4514915549394948, 1st task Loss: 0.05275246209031796, 2nd task Loss: 0.3987390928491768, Accuracy A: 98.0786825251601 %, B: 98.65884325230512 %, C: 81.32279534109817 %, D: 81.9267515923567 %
Epoch 2/10, Total Mean Loss: 0.2225341290171939, 1st task Loss: 0.0151351892167399, 2nd task Loss: 0.207398939800454, Accuracy A: 99.63403476669717 %, B: 99.74853310980721 %, C: 91.13976705490849 %, D: 92.63535031847134 %
Epoch 3/10, Total Mean Loss: 0.11796216161721111, 1st task Loss: 0.005192160839915145, 2nd task Loss: 0.11277000077729597, Accuracy A: 100.0 %, B: 99.91617770326907 %, C: 95.46589018302829 %, D: 96.73566878980891 %
Epoch 4/10, Total Mean Loss: 0.06889588694441862, 1st task Loss: 0.004277039631226331, 2nd task Loss: 0.06461884731319228, Accuracy A: 99.90850869167429 %, B: 99.91617770326907 %, C: 97.67054908485856 %, D: 98.72611464968153 %
Epoch 5/10, Total Mean Loss: 0.03428887519656484, 1st task Loss: 0.003215837827000106, 2nd task Loss: 0.0310730373

In [ ]:
%%time
test_loop_rehearsal(test_dataloader_ABCD, model_rehearsal, loss, device)

Total Mean Loss per images: 0.6091500777041774, 1st task Loss: 0.25782194642274353, 2nd task Loss: 0.3513281312814338 Accuracy A: 94.4206008583691 %, B: 92.37288135593221 %, C: 86.98010849909583 %, D: 91.04761904761905 %


A-GEM with gradients selection

In [ ]:
def project_gradient(g, g_ref):

    dot_product = torch.dot(g, g_ref)
    if dot_product >= 0:
        return g
    else:
    #g_update = g - ( (g^t * g_ref) / (g_ref^T*g_ref) ) * g_ref ;  here we don't apply the transpose because g is already a vector
        dot_product_g_ref = torch.dot(g_ref, g_ref) + 1e-10 # to avoid division by zero
        g_update = g - (dot_product / dot_product_g_ref) * g_ref
        return g_update

In [ ]:

def train_loop_gradient_selection_task0(model, dataloader, loss_fn, num_epoch, optimizer, device):

    model.train()
    model = model.to(device)

    #we want to store all the images with the norm of their gradient in memory to select the ones with the higher norm gradient to use it during the 2nd task and thus avoid forgetting
    data_class0 = [] #(image, gradient norm)
    data_class1 = [] #(image, gradient norm)

    for epoch in range(num_epoch):

        epoch_loss = 0.0

        for images, labels, tasks in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            tasks = tasks.to(device)

            optimizer.zero_grad()

            if (epoch < num_epoch - 1):
                outputs = model(images, tasks)

                loss_batch = loss_fn(outputs, labels)

                loss_batch.backward()
                optimizer.step()
                epoch_loss += loss_batch.item() * len(tasks) #if the last batch has a different size from the others



            else: # We will check the gradient norms only for the last epoch and for each image

                for i in range(images.size(0)):
                    optimizer.zero_grad()
                    image_i = images[i].unsqueeze(0)
                    label_i = labels[i].unsqueeze(0)
                    task_i = tasks[i].unsqueeze(0)

                    pred_i = model(image_i, task_i)
                    loss_i = loss_fn(pred_i, label_i)

                    loss_i.backward()

                    #calculate the norm of the gradient
                    sum_squar = 0.0
                    for param in model.heads.parameters():
                        if param.grad is not None:
                            sum_squar += torch.sum(param.grad ** 2).item()
                    norm = sum_squar ** 0.5

                    img = images[i].cpu().detach()
                    lbl= labels[i].cpu().detach()

                    if lbl == 0:
                        data_class0.append((img,norm ))
                    elif lbl== 1:
                        data_class1.append((img,norm ))

                optimizer.zero_grad()
                outputs = model(images, tasks)
                loss_batch = loss_fn(outputs, labels)
                loss_batch.backward()
                optimizer.step()
                epoch_loss += loss_batch.item()* len(tasks)
        epoch_loss = epoch_loss/len(dataloader.dataset) # we divide by the total number of images in the dataset


        print(f"Epoch {epoch+1}/{num_epoch}, Loss : {epoch_loss}")

    return data_class0, data_class1




In [ ]:
model_agem_grad_select = rehearsal()
model_agem_grad_select.add_task(0)
model_agem_grad_select.add_task(1)
optimizer_agem_grad_select= optim.SGD(model_agem_grad_select.parameters(), lr=0.001, momentum=0.9)

In [ ]:
data_class_0, data_class_1 = train_loop_gradient_selection_task0(model_agem_grad_select, train_dataloader_AB_full ,loss, num_epochs, optimizer_agem_grad_select, device)

Epoch 1/10, Loss : 0.29735867453956144
Epoch 2/10, Loss : 0.13120245244032233
Epoch 3/10, Loss : 0.08288440575732357
Epoch 4/10, Loss : 0.050001842834311215
Epoch 5/10, Loss : 0.03507266926304037
Epoch 6/10, Loss : 0.024187862999751925
Epoch 7/10, Loss : 0.015784437004244617
Epoch 8/10, Loss : 0.014270978931907019
Epoch 9/10, Loss : 0.009045667911309683
Epoch 10/10, Loss : 0.00837586349393238


In [ ]:
weights_AB = copy.deepcopy(model_agem_grad_select.state_dict()) # after we are going to train the task 1 on different parameters so we don't want to train again on the task 0 which is common for all the other models with different parameters for the parameters concern the train on the task 1 and not 0

In [ ]:

def select_k_percent(data_class0, data_class1, k_percent=0.05):
    #Take 2 lists of couples (image, norm gradient), sort the couples with the highest gradient norm and then select k% of the first sorted elements of the 2 lists and put the associated image in one list with the couples ( image, label)
    nb_0 = int(len(data_class0) * k_percent) # number of images of the 1st class we are going to store in memory
    nb_1 = int(len(data_class1) * k_percent) # number of images of the 2nd class we are going to store in memory
    data_k_percent = []

    data_class0.sort(key=lambda x: x[1], reverse=True) #reverse = True for the descending" order

    for couple in data_class0[:nb_0]:
        data_k_percent.append((couple[0], 0))


    data_class1.sort(key=lambda x: x[1], reverse=True)

    for couple in data_class1[:nb_1]:
        data_k_percent.append((couple[0], 1))


    return data_k_percent

In [ ]:
'''
def train_loop_gradient_selection_task1(model, dataloader, data_k_percent, loss_fn, num_epoch, optimizer, device, batch_size):
    model.train()
    model = model.to(device)

    for epoch in range(num_epoch):
        epoch_loss = 0.0
        epoch_loss_task0 = 0.0
        epoch_loss_task1 = 0.0
        dic_acc = {"A_true": 0,"A_tot": 0,  "B_true": 0, "B_tot" : 0,  "C_true": 0, "C_tot" : 0, "D_true": 0, "D_tot": 0}  # accuracy dictionary for each class

        for images, labels, tasks in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            tasks = tasks.to(device)

            optimizer.zero_grad()
            outputs_task1 = model(images, tasks)
            loss_task1 = loss_fn(outputs_task1, labels)
            loss_task1.backward()
            epoch_loss_task1 += loss_task1.item() * images.size(0)

            epoch_loss_task1 += loss_task1.item() * images.size(0)

            pred_class_t1 = outputs_task1.argmax(dim=1)

            mask_C = (labels == 0)
            dic_acc["C_true"] += (pred_class_t1[mask_C] == 0).sum().item()
            dic_acc["C_tot"] += mask_C.sum().item()

            mask_D = (labels == 1)
            dic_acc["D_true"] += (pred_class_t1[mask_D] == 1).sum().item()
            dic_acc["D_tot"] += mask_D.sum().item()

            g_new = [] #gradient of the task 1
            for param in model.parameters():
                if param.grad is not None:
                    g_new.append(param.grad.view(-1)) #flatten the vector
            g_new = torch.cat(g_new)

            if len(data_k_percent) >= batch_size :

                batch_task0 = random.sample(data_k_percent, batch_size)

            else :
                raise ValueError("the size of the batch for the task 0 can not be higher than the truncked dataset itself")

            mem_images = torch.stack([couple[0] for couple in batch_task0]).to(device)
            mem_labels = torch.tensor([couple[1] for couple in batch_task0]).to(device)
            mem_tasks = torch.zeros_like(mem_labels).to(device) # all the images in the sample memory are from the task 0


            model.zero_grad()

            outputs_mem = model(mem_images, mem_tasks)
            loss_mem = loss_fn(outputs_mem, mem_labels)
            loss_mem.backward()

            epoch_loss_task0 += loss_mem.item() * mem_images.size(0)
            total_mem_images_seen += mem_images.size(0)

            pred_class_t0 = outputs_mem.argmax(dim=1)

            mask_A = (mem_labels == 0)
            dic_acc["A_true"] += (pred_class_t0[mask_A] == 0).sum().item()
            dic_acc["A_tot"] += mask_A.sum().item()

            mask_B = (mem_labels == 1)
            dic_acc["B_true"] += (pred_class_t0[mask_B] == 1).sum().item()
            dic_acc["B_tot"] += mask_B.sum().item()

            g_ref = [] #gradient of the task 0
            for param in model.parameters():
                if param.grad is not None:
                    g_ref.append(param.grad.view(-1))
            g_ref = torch.cat(g_ref)

            g_update =project_gradient(g_new, g_ref)

            if not (g_update == g_ref):

                index = 0
                for param in model.parameters():
                    if param.grad is not None:
                        num_el = param.grad.numel()
                        # On redonne au vecteur plat sa forme d'origine (matrice de poids)
                        param.grad.copy_(g_update[index:index + num_el].view_as(param.grad))
                        index += num_el
            else:

                index = 0
                for param in model.parameters():
                    if param.grad is not None:
                        num_el = param.grad.numel()
                        param.grad.copy_(g_new[index:index + num_el].view_as(param.grad))
                        index += num_el


            optimizer.step()
            epoch_loss_task0 += loss_task0.item()* len(tasks)
            epoch_loss = epoch_loss / len(dataloader_task1.dataset)

        print(f'Total Mean Loss per images: {loss_test}, 1st task Loss: {loss_testAB}, 2nd task Loss: {loss_testCD}',
              f'Accuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')
'''

In [ ]:

def train_loop_gradient_selection_task_0_1(model, dataloader, data_k_percent, loss_fn, num_epoch, optimizer, device, batch_size):
    start_time = time.time()
    model.train()
    model = model.to(device)

    for epoch in range(num_epoch):

        epoch_loss_task0 = 0.0
        epoch_loss_task1 = 0.0
        total_mem_images_seen = 0

        dic_acc = {"A_true": 0, "A_tot": 0, "B_true": 0, "B_tot": 0, "C_true": 0, "C_tot": 0, "D_true": 0, "D_tot": 0}

        for images, labels, tasks in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            tasks = tasks.to(device)

            optimizer.zero_grad()
            outputs_task1 = model(images, tasks)
            loss_task1 = loss_fn(outputs_task1, labels)
            loss_task1.backward()

            epoch_loss_task1 += loss_task1.item() * images.size(0)


            pred_class_t1 = outputs_task1.argmax(dim=1)

            mask_C = (labels == 0)
            dic_acc["C_true"] += (pred_class_t1[mask_C] == 0).sum().item()
            dic_acc["C_tot"] += mask_C.sum().item()

            mask_D = (labels == 1)
            dic_acc["D_true"] += (pred_class_t1[mask_D] == 1).sum().item()
            dic_acc["D_tot"] += mask_D.sum().item()

            for param in model.parameters():
                if param.grad is None and param.requires_grad:
                    param.grad = torch.zeros_like(param.data)

            g_new = []
            for param in model.parameters():
                if param.grad is not None:
                    g_new.append(param.grad.view(-1))
            g_new = torch.cat(g_new)

            if len(data_k_percent) >= batch_size:
                batch_task0 = random.sample(data_k_percent, batch_size)
            else:
                raise ValueError("the size of the batch for the task 0 can not be higher than the truncked dataset itself")

            mem_images = torch.stack([couple[0] for couple in batch_task0]).to(device)
            mem_labels = torch.tensor([couple[1] for couple in batch_task0]).to(device)
            mem_tasks = torch.zeros_like(mem_labels).to(device)

            model.zero_grad()

            outputs_mem = model(mem_images, mem_tasks)
            loss_mem = loss_fn(outputs_mem, mem_labels)
            loss_mem.backward()

            epoch_loss_task0 += loss_mem.item() * mem_images.size(0)
            total_mem_images_seen += mem_images.size(0)

            pred_class_t0 = outputs_mem.argmax(dim=1)

            mask_A = (mem_labels == 0)
            dic_acc["A_true"] += (pred_class_t0[mask_A] == 0).sum().item()
            dic_acc["A_tot"] += mask_A.sum().item()

            mask_B = (mem_labels == 1)
            dic_acc["B_true"] += (pred_class_t0[mask_B] == 1).sum().item()
            dic_acc["B_tot"] += mask_B.sum().item()


            for param in model.parameters():
                if param.grad is None and param.requires_grad:
                    param.grad = torch.zeros_like(param.data)

            g_ref = []
            for param in model.parameters():
                if param.grad is not None:
                    g_ref.append(param.grad.view(-1))
            g_ref = torch.cat(g_ref)


            g_update = project_gradient(g_new, g_ref)

            index = 0
            for param in model.parameters():
                if param.grad is not None:
                    num_el = param.grad.numel()
                    param.grad.copy_(g_update[index:index + num_el].view_as(param.grad))
                    index += num_el

            optimizer.step()

        final_loss_t1 = epoch_loss_task1 / len(dataloader.dataset)
        final_loss_t0 = epoch_loss_task0 / total_mem_images_seen if total_mem_images_seen > 0 else 0.0
        total_mean_loss = final_loss_t1 + final_loss_t0

        print(f'Epoch {epoch+1}/{num_epoch} - Total Mean Loss: {total_mean_loss:.4f}, 1st task Loss: {final_loss_t0}, 2nd task Loss: {final_loss_t1:.4f}',
              f'\nAccuracy A: {(dic_acc["A_true"] * 100 / dic_acc["A_tot"] if dic_acc["A_tot"] > 0 else 0.0)} %, '
              f'B: {(dic_acc["B_true"] * 100 / dic_acc["B_tot"] if dic_acc["B_tot"] > 0 else 0.0)} %, '
              f'C: {(dic_acc["C_true"] * 100 / dic_acc["C_tot"] if dic_acc["C_tot"] > 0 else 0.0)} %, '
              f'D: {(dic_acc["D_true"] * 100 / dic_acc["D_tot"] if dic_acc["D_tot"] > 0 else 0.0)} %')

    end_time = time.time()
    real_time = end_time - start_time
    return real_time

In [ ]:
batch_size = 30

In [ ]:
time = []
loss_task0 = []
accuracy = []

In [ ]:
# we want to study the impact of the size of the sample of the task 0 on the final results

sample_task0_5 = select_k_percent(data_class_0, data_class_1, k_percent=0.05)
sample_task0_10 = select_k_percent(data_class_0, data_class_1, k_percent=0.5)
sample_task0_15 = select_k_percent(data_class_0, data_class_1, k_percent=0.15)
sample_task0_20 = select_k_percent(data_class_0, data_class_1, k_percent=0.2)

print(len(sample_task0_5))
print(len(sample_task0_20))

In [ ]:
time.happend(train_loop_gradient_selection_task_0_1(model_agem_grad_select, train_dataloader_ABCD, sample_task0_5, loss, num_epochs, optimizer_agem_grad_select, device, batch_size))


Epoch 1/10 - Total Mean Loss: 0.3232, 1st task Loss: 0.029201783798222917, 2nd task Loss: 0.2940 
Accuracy A: 99.00621118012423 %, B: 99.38202247191012 %, C: 85.67343437231914 %, D: 86.58569500674764 %
Epoch 2/10 - Total Mean Loss: 0.1913, 1st task Loss: 0.028810662223619565, 2nd task Loss: 0.1625 
Accuracy A: 99.13419913419914 %, B: 99.0975747320925 %, C: 93.10837861023735 %, D: 94.0080971659919 %
Epoch 3/10 - Total Mean Loss: 0.1357, 1st task Loss: 0.02481687436729208, 2nd task Loss: 0.1109 
Accuracy A: 99.56521739130434 %, B: 99.21348314606742 %, C: 95.02430654847012 %, D: 96.57219973009447 %
Epoch 4/10 - Total Mean Loss: 0.0876, 1st task Loss: 0.018622953613676066, 2nd task Loss: 0.0690 
Accuracy A: 99.51159951159951 %, B: 99.77168949771689 %, C: 97.56934515298828 %, D: 98.13765182186235 %
Epoch 5/10 - Total Mean Loss: 0.0600, 1st task Loss: 0.020818638314484758, 2nd task Loss: 0.0392 
Accuracy A: 99.62825278810409 %, B: 99.49324324324324 %, C: 99.05633400057192 %, D: 99.4062078272

In [ ]:
l, a = test_loop_rehearsal(test_dataloader_ABCD, model_agem_grad_select, loss, device) # we can use the test function of rehearsal
loss_task0.happend(l)
accuracy.happend(a)

Total Mean Loss per images: 0.6315094779461643, 1st task Loss: 0.2013264275662331, 2nd task Loss: 0.43018305037993115 Accuracy A: 95.27896995708154 %, B: 91.52542372881356 %, C: 84.99095840867993 %, D: 91.80952380952381 %


Impact of the size of the samples

In [ ]:
model_agem_grad_select.load_state_dict(weights_AB)

In [ ]:
time.happend(train_loop_gradient_selection_task_0_1(model_agem_grad_select, train_dataloader_ABCD, sample_task0_10, loss, num_epochs, optimizer_agem_grad_select, device, batch_size))


In [ ]:
l,a = test_loop_rehearsal(test_dataloader_ABCD, model_agem_grad_select, loss, device)
loss_task0.happend(l)
accuracy.happend(a)

In [ ]:
model_agem_grad_select.load_state_dict(weights_AB)

In [ ]:
time.happend(train_loop_gradient_selection_task_0_1(model_agem_grad_select, train_dataloader_ABCD, sample_task0_15, loss, num_epochs, optimizer_agem_grad_select, device, batch_size))


In [ ]:
l,a = test_loop_rehearsal(test_dataloader_ABCD, model_agem_grad_select, loss, device)
loss_task0.happend(l)
accuracy.happend(a)

In [ ]:
model_agem_grad_select.load_state_dict(weights_AB)

In [ ]:
time.happend(train_loop_gradient_selection_task_0_1(model_agem_grad_select, train_dataloader_ABCD, sample_task0_20, loss, num_epochs, optimizer_agem_grad_select, device, batch_size))


In [ ]:
l,a = test_loop_rehearsal(test_dataloader_ABCD, model_agem_grad_select, loss, device)
loss_task0.happend(l)
accuracy.happend(a)

Visualization

In [ ]:
pourcent = [5, 10, 15, 20]

acc_A = []
acc_B = []
for d in accuracy:
    acc_A.happend((d["A_true"] / d["A_tot"] * 100) if d["A_tot"] > 0 else 0.0)
    acc_B.happend((d["B_true"] / d["B_tot"] * 100) if d["B_tot"] > 0 else 0.0)


fig, axs = plt.subplots(1, 3, figsize=(18, 5))

#Time
axs[0].plot(pourcent, time, marker='o', color='g', linestyle='-', linewidth=2)
axs[0].set_title("Execution time based on pourcent")
axs[0].set_xlabel(" % of each class we take from the task 0")
axs[0].set_ylabel("Time (s)")
axs[0].set_xticks("pourcent")
axs[0].grid(True, linestyle='--', alpha=0.6)

# Loss Task 0
axs[1].plot(pourcent, loss_task0, marker='s', color='r', linestyle='-', linewidth=2)
axs[1].set_title("Loss Task 0 based on pourcent")
axs[1].set_xlabel("% of each class we take from the task 0")
axs[1].set_ylabel("Loss")
axs[1].set_xticks("pourcent")
axs[1].grid(True, linestyle='--', alpha=0.6)

# Accuracy A and B during the train of the task 1
x = np.arange(len(pourcent))
width = 0.35  # Largeur de chaque barre

bar1 = axs[2].bar(x - width/2, acc_A, width, label='Buildings', color='#1f77b4')
bar2 = axs[2].bar(x + width/2, acc_B, width, label='Street', color='#aec7e8')

axs[2].set_title("Accuracy of buildings and street based on porcent")
axs[2].set_xlabel("% of each class we take from the task 0")
axs[2].set_ylabel("Accuracy (%)")
axs[2].set_xticks(x)
axs[2].set_xticklabels([f"{t}%" for t in pourcent])
axs[2].set_ylim(0, 105)
axs[2].legend()
axs[2].grid(True, axis='y', linestyle='--', alpha=0.6)


plt.tight_layout()


plt.show()

NameError: name 'accuracy' is not defined

In [ ]:
### A partir d'ici c'set la partie 2 du cod . Avant ici c'est la partoe 1 du code . La partie 2 se foculise ecxcluisiovement sur les methodes d'entrainement eavec A-GEM . La partie 1 se focalise sur les prerequis pour la constructui des datstatset , dataloader et autrse et surtout sur le continual leanring sans A -GEM

In [ ]:


classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}
# To do the reverse path
label_to_classname= {0: 'buildings', 1: 'forest', 2: 'glacier', 3: 'mountain', 4: 'sea', 5: 'street'}

class IntelDataset(Dataset):
    def __init__(self, df, transform=None, base_dir=None, label_map=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir = base_dir # This will now be the task-specific img folder
        self.label_map = label_map # Conversion dictionary, ex: {3: 1, 1: 0}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        label = int(row['label'])

        # No need to get class_name from label_to_classname if base_dir is already specific
        # The base_dir should be something like /content/my_intel_dataset/seg_train/seg_train/df_buildings_sea/img

        # Remapping so the label is 0 or 1 for the binary model
        if self.label_map is not None:
            label = self.label_map[label]

        # Simplified img_path construction
        img_path = os.path.join(self.base_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
#### GLOBAL PARAMETERS FOR PART 2 ####

NB_EPOCH=3
epochs_per_task=2
memory_size_per_task=300
batch_size=32

classe = {'buildings' : 0, 'forest' : 1, 'glacier' : 2, 'mountain' : 3, 'sea' : 4, 'street' : 5}
device="cuda" if torch.cuda.is_available() else "cpu"

# Mapping for each task to local head indices [0, 1]
task_mappings = {
    0: {0: 0, 4: 1},  # Task 0: Buildings -> 0, Sea -> 1
    1: {1: 0, 3: 1}   # Task 1: Forest -> 0, Mountain -> 1
}

tasks_definition = {
    0: [0, 4],
    1: [1, 3]
}

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

def create_mapped_loader(df, task_id, root_dir):

    # I use Laura's transform pipeline in my part (2e part)
    ds = IntelDataset(df, transform=transform, base_dir=root_dir, label_map=None)
    return DataLoader(ds, batch_size=32, shuffle=False)

#Dictionary containing the good dataloader for the test
dict_test_loader = {
    0: create_mapped_loader(label_AB_test, 0, path_img_AB_test),
    1: create_mapped_loader(label_CD_test, 1, path_img_CD_test),
}

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import KMeans


class EpisodicMemory:
  def __init__(self, max_memories_sample=1000):
    """
    max_memory_sample : the max images to keep for each past task
    """
    self.max_memories_sample=max_memories_sample
    self.memory_dataset=None   # No TensorDataset yet

  def _extract_features(self,model,dataset,device):
    """
    extract the features of the layer just before the last layer
    """
    # switch to eval mode
    model.eval()

    # We take as feature extractor the whole model except the last layer
    backbone = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
    features_list=[]
    temp_loader=DataLoader(dataset, batch_size=64,shuffle=False)

    with torch.no_grad():
      for images, _ in temp_loader:
        images=images.to(device)
        # the features go into the backbone to have the features
        features=backbone(images)
        features = features.view(features.size(0), -1)
        features_list.append(features.cpu().numpy())

    model.train()
    return np.concatenate(features_list, axis=0)

  def KMeans_sample_extractor(self,task_dataset, model=None,device='cuda'):
    """
    We extract the feature to choose the best image for each cluster using KMeans
    """
    budget = min(self.max_memories_sample, len(task_dataset))

    if model is not None:
      print("Extraction of the features with KMeans")
      features=self._extract_features(model,task_dataset,device)

      print(f'We use KMeans with K={budget}')

      # Use of KMEans to create clusters
      kmeans = KMeans(n_clusters=budget, n_init='auto', random_state=42)
      cluster_labels = kmeans.fit_predict(features)
      # the centroids
      centroids = kmeans.cluster_centers_

      # Selection of the best image for each cluster
      selected_indices=[]
      for i in range(budget):
        idx_in_cluster = np.where(cluster_labels == i)[0]
        if len(idx_in_cluster) > 0:
          cluster_features = features[idx_in_cluster]

          # We compute the distances
          distances = np.linalg.norm(cluster_features - centroids[i], axis=1)
          # The best indices
          best_local_idx = np.argmin(distances)
          selected_indices.append(idx_in_cluster[best_local_idx])


      # extraction of the tensors (images,labels) linked to these selected indices
      full_loader = DataLoader(task_dataset, batch_size=len(task_dataset), shuffle=False)
      all_images, all_labels = next(iter(full_loader))

      # Final selected images and labels
      images_from_samples = all_images[selected_indices]
      labels_sample = all_labels[selected_indices]

      print("K-Means ended")

      return images_from_samples,labels_sample


  def store_memories(self,task_dataset,id_task,method="random"):
    """
    extracts a random sample from the task that has just finished
    Method: “random” for the standard random method, “KMeans” for the K-means method
    """
    if method=="KMeans":
      images_from_samples,labels_sample=self.KMeans_sample_extractor(task_dataset,model,device)

    else:
      budget = min(self.max_memories_sample, len(task_dataset))

      # We create a temporary dataloader with batchsize=self.max_memories_sample
      temp_loader=DataLoader(task_dataset, batch_size=budget,shuffle=True)
      images_from_samples,labels_sample = next(iter(temp_loader))

    task_ids = torch.full((len(labels_sample),), id_task, dtype=torch.long)


    if self.memory_dataset is None:

      # Merge the images and the labels in a tensordataset
      self.memory_dataset=TensorDataset(images_from_samples,labels_sample, task_ids)

    else:
      old_labels=self.memory_dataset.tensors[1]
      old_images=self.memory_dataset.tensors[0]
      old_task_ids = self.memory_dataset.tensors[2]

      # We merge the past and the present

      new_images=torch.cat([old_images,images_from_samples] )
      new_labels=torch.cat([old_labels,labels_sample])
      new_task_ids = torch.cat([old_task_ids, task_ids])

      self.memory_dataset=TensorDataset(new_images,new_labels,new_task_ids)

      print(f"-> A-GEM Buffer updated. Total memory size: {len(self.memory_dataset)} images.")



  def get_reference_batch(self, batch_size):
    if self.memory_dataset is None:
      return None

    ref_loader=DataLoader(self.memory_dataset, batch_size=batch_size,shuffle=True)
    return next(iter(ref_loader))


In [ ]:
def train_agrem_step(model, optimizer, criterion, current_batch, batch_ref, device, id_task):
    """
    Train the model using A-GEM for one step, handling Multi-Head gradients securely.
    """
    current_imgs, current_labels = current_batch
    current_imgs, current_labels = current_imgs.to(device), current_labels.to(device)


    img_ref, label_ref, task_id_ref = batch_ref
    img_ref, label_ref, task_id_ref = img_ref.to(device), label_ref.to(device), task_id_ref.to(device)

    # 1. Compute g (gradient for the current task)
    optimizer.zero_grad()
    current_output = model(current_imgs, id_task)
    current_loss = criterion(current_output, current_labels)
    current_loss.backward()

    # We replace the 'None by zeros' for more security
    grads = []
    for p in model.parameters():
        if p.requires_grad:
            if p.grad is not None:
                grads.append(p.grad.view(-1).clone())
            else:
                grads.append(torch.zeros_like(p).view(-1))
    g = torch.cat(grads)

    # 2. Compute g_ref (gradient for memory/reference batch)
    optimizer.zero_grad()

    #It prevent problems with the BathNorm (corruption of the memory)
    model.eval()

    # We calculate the loss
    unique_tasks = torch.unique(task_id_ref)
    loss_ref = 0
    for t in unique_tasks:
        mask = (task_id_ref == t)
        img_t = img_ref[mask]
        label_t = label_ref[mask]

        output_t = model(img_t, id_task=t.item())
        loss_t = criterion(output_t, label_t)
        # Pondération de la loss selon le nombre d'images de cette tâche
        loss_ref += loss_t * (img_t.size(0) / img_ref.size(0))

    loss_ref.backward()

    # Return in train mode
    model.train()

    grads_ref = []
    for p in model.parameters():
        if p.requires_grad:
            if p.grad is not None:
                grads_ref.append(p.grad.view(-1).clone())
            else:
                grads_ref.append(torch.zeros_like(p).view(-1))
    g_ref = torch.cat(grads_ref)


    # I use Laura's function to do A-GEM Projection
    g_update=project_gradient(g,g_ref)
    g=g_update

    # 4. Manually update gradients in the model
    optimizer.zero_grad()
    index = 0
    for p in model.parameters():
        if p.requires_grad:
            num_params = p.numel()
            p.grad = g[index:index + num_params].view(p.shape).clone()
            index += num_params

    optimizer.step()
    return current_loss.item()

In [ ]:
def train_standard_step(model, optimizer, criterion, current_batch, device, id_task):
  """
  Train the model WITHOUT using A-GEM model for one step.
  """
  current_imgs, current_labels = current_batch
  current_imgs, current_labels = current_imgs.to(device), current_labels.to(device)

  optimizer.zero_grad()
  predictions = model(current_imgs, id_task)
  loss = criterion(predictions, current_labels)
  loss.backward()
  optimizer.step()

  return loss.item()

In [ ]:
from sklearn.metrics import confusion_matrix,recall_score

def evaluate_continual_eval(model, test_loaders, list_of_learned_task, device):
    """
    Function to evaluate the performances : accuracy and loss on the test set
    """
    model.eval()
    results = {}

    with torch.no_grad():
        for id_task in list_of_learned_task:
            specific_test_loader = test_loaders[id_task]
            total_correct = 0
            total_loss = 0.0
            total_samples = 0

            # List to store the current result

            all_preds = []
            all_labels = []

            for images, labels in specific_test_loader:
                images, labels = images.to(device), labels.to(device)

                logits = model(images, id_task)
                loss = criterion(logits, labels)

                _, preds = torch.max(logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                total_correct += (preds == labels).sum().item()
                total_loss += loss.item() * labels.size(0)
                total_samples += labels.size(0)

            # For the confusion matrix
            cm = confusion_matrix(all_labels, all_preds)
            #For the recalls
            recalls = recall_score(all_labels, all_preds, average=None, zero_division=0)



            #Dict with the performance
            results[id_task] = {
                'acc': total_correct / total_samples,
                'loss': total_loss / total_samples,
                "cm":cm,
                'recall': recalls
            }

    model.train()
    return results

In [ ]:
from tqdm import tqdm

def train_agem_global(model, optimizer, criterion, dict_test_loader, device='cuda'):
  """
  Final function to train the model with the A-GEM method (for all the step)
  """

  # Initialize memory buffer to store samples from previous tasks
  buffer_agem = EpisodicMemory(max_memories_sample=memory_size_per_task)
  list_of_learned_task = []
  initial_lr = 0.001

  model.to(device)

  for id_task, classes in tasks_definition.items():

    for param_group in optimizer.param_groups:
      param_group['lr'] = initial_lr

    # set gammma to 1 to cancel its effect
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    print(f"Learning Rate reinitialised at : {optimizer.param_groups[0]['lr']}")

    print(f"\n--- Task {id_task} (Classes: {classes}) ---")

    #We do Local mapping: map original classes to 0 and 1 for the specific task head
    local_map = {classes[0]: 0, classes[1]: 1}

    # We Select only images belonging to the current task classes
    #current_df = label_ABCD_train[label_ABCD_train['label'].isin(classes)]
    current_df = label_ABCD_train[label_ABCD_train['task_id'] == id_task]

    # Determine the correct image base directory for the current task
    if id_task == 0:
        current_task_img_path = path_img_AB_train
    elif id_task == 1:
        current_task_img_path = path_img_CD_train
    else:
        raise ValueError(f"Unknown task ID: {id_task}")


    current_dataset = IntelDataset(
        current_df,
        transform=transform,
        label_map=None,
        base_dir=current_task_img_path # Use the task-specific path here
    )
    current_loader = DataLoader(current_dataset, batch_size=batch_size, shuffle=True)
    list_of_learned_task.append(id_task)

    for epoch in range(1, epochs_per_task + 1):
      model.train()
      for imgs, labels in tqdm(current_loader, desc=f"Epoch {epoch}"):
        # Get a reference batch from the memory buffer
        batch_ref = buffer_agem.get_reference_batch(batch_size=batch_size)

        if batch_ref is not None:

          imgs_ref, labels_ref, task_ids_ref = batch_ref

          # Use A-GEM step if we have samples from previous tasks

          train_agrem_step(model, optimizer, criterion, (imgs, labels), (imgs_ref, labels_ref,task_ids_ref), device, id_task)
        else:
          # Standard training for the first task
          train_standard_step(model, optimizer, criterion, (imgs, labels), device, id_task)

      print(f"\n>>> EVALUATION AFTER EPOCH {epoch} (Task {id_task}) <<<")
      results = evaluate_continual_eval(model, dict_test_loader, list_of_learned_task, device)

      for t_id, metrics in results.items():
          c_ids = tasks_definition[t_id]
          name0, name1 = label_to_classname[c_ids[0]], label_to_classname[c_ids[1]]
          print(f"  * Task {t_id} Performance :")
          print(f"    - Accuracy : {metrics['acc']*100:.2f}% | Loss : {metrics['loss']:.4f}")
          print(f"    - Recall {name0} (0): {metrics['recall'][0]*100:.2f}%")
          print(f"    - Recall {name1} (1): {metrics['recall'][1]*100:.2f}%")
          print(f"    - Confusion Matrix :\n{metrics['cm']}")
      print("-" * 50)
      # =======================================================================
      scheduler.step()
      new_lr = optimizer.param_groups[0]['lr']
      print(f"Learning Rate updated : {new_lr}")


    # Save some images to the buffer after finishing the task
    buffer_agem.store_memories(current_dataset,id_task=id_task ,method='KMeans')


In [ ]:
# Clear GPU cache
torch.cuda.empty_cache()

class MultiHeadResNet(nn.Module):
    def __init__(self, original_resnet):
        super(MultiHeadResNet, self).__init__()
        # Backbone: all layers except the final FC
        self.features = nn.Sequential(*list(original_resnet.children())[:-1])
        # One head (Linear layer) per task, each outputting 2 classes
        self.heads = nn.ModuleList([nn.Linear(512, 2), nn.Linear(512, 2)])

    def forward(self, x, id_task):
        # Changed parameter name task_id to id_task to fix the TypeError
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.heads[id_task](x)
        return x

# Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model = MultiHeadResNet(base_model).to(device)

criterion = nn.CrossEntropyLoss()
# Optimizer tracks all parameters; A-GEM step handles the gradient logic
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Start training
train_agem_global(model, optimizer, criterion, dict_test_loader, device)


--- Task 0 (Classes: [0, 4]) ---


Epoch 2: 100%|██████████| 143/143 [01:57<00:00,  1.22it/s]



 Result after task 0 :
   * Task 0 -> Accuracy: 92.64% | Loss: 0.1631

--- Task 1 (Classes: [1, 3]) ---


Epoch 2: 100%|██████████| 154/154 [04:47<00:00,  1.87s/it]


-> A-GEM Buffer updated. Total memory size: 600 images.

 Result after task 1 :
   * Task 0 -> Accuracy: 92.64% | Loss: 0.1865
   * Task 1 -> Accuracy: 88.59% | Loss: 0.2709


In [ ]:
# Clear GPU cache
torch.cuda.empty_cache()

class MultiHeadResNet(nn.Module):
    def __init__(self, original_resnet):
        super(MultiHeadResNet, self).__init__()
        # Backbone: all layers except the final FC
        self.features = nn.Sequential(*list(original_resnet.children())[:-1])
        # One head (Linear layer) per task, each outputting 2 classes
        self.heads = nn.ModuleList([nn.Linear(512, 2), nn.Linear(512, 2)])

    def forward(self, x, id_task):
        # Changed parameter name task_id to id_task to fix the TypeError
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.heads[id_task](x)
        return x

# Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model = MultiHeadResNet(base_model).to(device)

criterion = nn.CrossEntropyLoss()
# Optimizer tracks all parameters; A-GEM step handles the gradient logic
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Start training
train_agem_global(model, optimizer, criterion, dict_test_loader, device)

Learning Rate reinitialised at : 0.001

--- Task 0 (Classes: [0, 4]) ---


Epoch 1: 100%|██████████| 143/143 [00:15<00:00,  9.25it/s]



>>> EVALUATION AFTER EPOCH 1 (Task 0) <<<
  * Task 0 Performance :
    - Accuracy : 92.00% | Loss : 0.1822
    - Recall buildings (0): 94.28%
    - Recall sea (1): 90.02%
    - Confusion Matrix :
[[412  25]
 [ 50 451]]
--------------------------------------------------
Learning Rate updated : 0.001


Epoch 2: 100%|██████████| 143/143 [00:14<00:00, 10.18it/s]



>>> EVALUATION AFTER EPOCH 2 (Task 0) <<<
  * Task 0 Performance :
    - Accuracy : 93.71% | Loss : 0.1677
    - Recall buildings (0): 91.76%
    - Recall sea (1): 95.41%
    - Confusion Matrix :
[[401  36]
 [ 23 478]]
--------------------------------------------------
Learning Rate updated : 0.001


...


>>> EVALUATION AFTER EPOCH 8 (Task 1) <<<
  * Task 0 Performance :
    - Accuracy : 91.47% | Loss : 0.2383
    - Recall buildings (0): 86.96%
    - Recall sea (1): 95.41%
    - Confusion Matrix :
[[380  57]
 [ 23 478]]
  * Task 1 Performance :
    - Accuracy : 89.61% | Loss : 0.3494
    - Recall forest (0): 88.79%
    - Recall mountain (1): 90.48%
    - Confusion Matrix :
[[491  62]
 [ 50 475]]
--------------------------------------------------
Learning Rate updated : 0.0005


Epoch 9: 100%|██████████| 154/154 [00:26<00:00,  5.73it/s]



>>> EVALUATION AFTER EPOCH 9 (Task 1) <<<
  * Task 0 Performance :
    - Accuracy : 91.15% | Loss : 0.2460
    - Recall buildings (0): 85.81%
    - Recall sea (1): 95.81%
    - Confusion Matrix :
[[375  62]
 [ 21 480]]
  * Task 1 Performance :
    - Accuracy : 89.52% | Loss : 0.3558
    - Recall forest (0): 88.97%
    - Recall mountain (1): 90.10%
    - Confusion Matrix :
[[492  61]
 [ 52 473]]
--------------------------------------------------
Learning Rate updated : 0.0005


Epoch 10: 100%|██████████| 154/154 [00:27<00:00,  5.68it/s]



>>> EVALUATION AFTER EPOCH 10 (Task 1) <<<
  * Task 0 Performance :
    - Accuracy : 91.68% | Loss : 0.2330
    - Recall buildings (0): 87.87%
    - Recall sea (1): 95.01%
    - Confusion Matrix :
[[384  53]
 [ 25 476]]
  * Task 1 Performance :
    - Accuracy : 89.42% | Loss : 0.3717
    - Recall forest (0): 86.08%
    - Recall mountain (1): 92.95%
    - Confusion Matrix :
[[476  77]
 [ 37 488]]
--------------------------------------------------
Learning Rate updated : 0.00025
Extraction of the features with KMeans
We use KMeans with K=300
K-Means ended
-> A-GEM Buffer updated. Total memory size: 600 images.



--- Task 0 (Classes: [0, 4]) ---


Epoch 6: 100%|██████████| 70/70 [00:37<00:00,  1.87it/s]



 Result after task 0 :
   * Task 0 -> Precision: 46.15% | Loss: 0.0172

--- Task 1 (Classes: [1, 3]) ---


Epoch 6: 100%|██████████| 75/75 [01:01<00:00,  1.23it/s]


-> A-GEM Buffer updated. Total memory size: 100 images.

 Result after task 1 :
   * Task 0 -> Precision: 46.15% | Loss: 0.0624
   * Task 1 -> Precision: 52.55% | Loss: 0.0190
